# Несколько блэкбордов

In [1]:
import os
from tqdm import tqdm
from src import Store

federal_path = './data/documents/1. Федеральный уровень'
regional_path = './data/documents/2. Региональный уровень'
local_path = './data/documents/3. Местный уровень'

store = Store()

for dir in [federal_path, regional_path, local_path]:
    for file in tqdm(os.listdir(dir)):
        path = os.path.join(dir, file)
        if 'НГП' in file:
            store.add_document(path)
        if 'СТП' in file:
            store.add_document(path)
        if ('ССЭР' in file):
            store.add_document(path)

100%|██████████| 2/2 [00:26<00:00, 13.14s/it]


In [2]:
from src.blackboard import BlackBoard

QUESTIONS = {
    'Социально-экономический анализ': [
        'Демография',
        'Производительность труда',
        'Структура и динамика ВГП',
        'Рынок труда',
        'Рынок жилья и коммерческой недвижимости',
        'Бюджетная обеспеченность'
    ],
    'Пространственный анализ': [
        'функциональная организация территории',
        'структура землевладения',
        'природно-экологический каркас',
        'архитектурно-градостроительная среда'
    ],
    'Транспортный анализ': [
        'Городской и внешний пассажирский и грузовой транспорт',
        'Парковки',
        'Пешеходные зоны',
    ],
    'Анализ инженерной инфраструктуры': [
        'Водоснабжение и водоотведение',
        'Энергоснабжение',
        'Теплоснабжение',
    ]
}

In [4]:
from src.tools import ddgs_tool
from src.agent import Agent
from src.llms import llm
from langchain.messages import SystemMessage, HumanMessage

from pydantic import BaseModel, Field

class PlannerResponse(BaseModel):
    questions : list[str] = Field(min_length=1, max_length=5, description='Перечень вопросов для агентов')

PLANNER_PROMPT = """
Ты — агент-планировщик. 
Твоя задача — разбить запрос пользователя на конкретные, атомарные подзадачи для выполнения другими агентами.
---
Правила работы:
1. Каждая подзадача должна быть сформулирована как четкий вопрос
2. Подзадачи должны быть атомарными — каждая охватывает ровно один аспект
3. Подзадачи должны быть независимыми (результат одной не требуется для другой)
4. Учитывай основную категорию и подкатегорию запроса — детализируй подкатегорию на подподкатегории
---
Примеры:
- запрос пользователя: демография
- подзадачи: 
    - сколько населения проживает в X?
    - какие показатели миграции в X?
    - ...
---
Запрос пользователя:
{query}
"""

RAG_PROMPT = """
Ты — агент, который отвечает на вопросы строго по данным из инструментов.

ЖЕСТКИЕ ПРАВИЛА (НАРУШЕНИЯ ЗАПРЕЩЕНЫ):

1. МАКСИМУМ 3 ВЫЗОВА ИНСТРУМЕНТОВ
   - После 3-го вызова ТЫ ОБЯЗАН немедленно сформировать ответ
   - Даже если данные неполные — отвечай тем, что есть
   - НЕ ПЫТАЙСЯ сделать 4-й вызов

2. НЕ ИСПОЛЬЗУЙ общие знания
   - Только информация из результатов вызовов инструментов
   - Если данных нет — так и напиши

3. НЕ ПОВТОРЯЙ вызовы с похожими параметрами
   - Если запрос не дал результатов — меняй подход, а не перефразируй

ПРИНУДИТЕЛЬНАЯ ОСТАНОВКА:
После каждого вызова считай: "Вызов №X из 3"
Когда X = 3 — СРАЗУ ПЕРЕХОДИ К ФОРМИРОВАНИЮ ОТВЕТА

---
Контекст:
{query}
---
Вопрос:
{question}
"""

SUMMARIZER_MESSAGE = """
Сформируй итоговый ответ на запрос пользователя, используя только ответы других агентов.

ОБЯЗАТЕЛЬНЫЕ ПРАВИЛА:
- Строго следуй формату запроса пользователя
- Можно использовать таблицы и структурированные блоки — это допустимо и приветствуется
- Не добавляй новые данные, прогнозы или внешние источники
- Не делай предположений и не заполняй пробелы самостоятельно
- Не упоминай процесс анализа, источники, записи или ход рассуждений
- Не добавляй рекомендации, планы действий или списки “что нужно сделать”
- Не комментируй полноту или неполноту данных

СОДЕРЖАНИЕ:
- Используй только факты, которые уже присутствуют во входных данных
- Если часть информации отсутствует — просто не включай её

СТИЛЬ:
- Развёрнутый аналитический ответ
- Можно использовать таблицы и разделы
- Без коротких ответов и без телеграфного стиля
- Без лишней воды
                           
Ответ должен быть финальным и не содержать комментариев о процессе его получения.
                           
ЗАПРЕЩЕНО:
- выводить любые данные, которых нет во входе
- делать прогнозы
- добавлять внешние знания
---
Запрос пользователя:
{query}
---
Предоставленная информация:
{answers}
"""

def ask_question(query : str, question : str) -> list[str]:
    system_prompts = RAG_PROMPT.format(query=query, question=question)
    web_agent = Agent(system_prompt=system_prompts, tools=[ddgs_tool])
    doc_agent = Agent(system_prompt=system_prompts, tools=[store.tool])
    answers = []
    for agent in [web_agent, doc_agent]:
        answer = agent.invoke([])
        answers.append(answer)
    return answers

def ask_query(query : str):
    planner_agent = Agent(PLANNER_PROMPT.format(query=query), response_format=PlannerResponse)
    questions = planner_agent.invoke([]).questions

    for question in questions:
        print(question)

    answers = []
    for question in tqdm(questions):
        results = ask_question(query, question)
        answers.extend(results)

    return llm.invoke(SUMMARIZER_MESSAGE.format(
        query=query,
        answers=answers
    ))

In [5]:
template = '{category} г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: {subcategory}'

queries = []

for category,subcategories in QUESTIONS.items():
    for subcategory in subcategories:
        query = template.format(category=category, subcategory=subcategory)
        queries.append(query)

results = {}

In [8]:
for query in queries:
    if query in results:
        continue
    print(query)
    try:
        result = ask_query(query)
        results[query] = result
    except:
        continue

Социально-экономический анализ г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: Производительность труда
Какой общий показатель производительности труда (ВВП на одного занятого) в г. Гатчина за последние 5 лет?
Как изменялась динамика производительности труда в г. Гатчина в разрезе основных отраслей (промышленность, строительство, услуги, сельское хозяйство) за тот же период?
Какой уровень средней заработной платы в г. Гатчина и как он коррелирует с показателем производительности труда?
Какой уровень занятости (число занятых) в г. Гатчина и как он соотносится с изменениями производительности труда?
Какие основные факторы (уровень образования, квалификация, автоматизация, инвестиции в основной капитал) влияют на производительность труда в г. Гатчина согласно региональным исследованиям и статистике ЛОКАЛЬНЫХ источников (Росстат, Ленстат, муниципальная статистика)?


100%|██████████| 5/5 [04:39<00:00, 55.89s/it]


Социально-экономический анализ г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: Бюджетная обеспеченность
Пространственный анализ г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: природно-экологический каркас
Какие типы землепользования (лесные массивы, сельскохозяйственные угодья, городская застройка, пустыни, водные объекты) распределены на территории г. Гатчина и Гатчинского муниципального района?
Какие охраняемые природные территории (заповедники, национальные парки, заказники, памятники природы) находятся в границах г. Гатчина и Гатчинского муниципального района?
Какова площадь и геометрия основных водных объектов (реки, озёра, пруды, болота) в г. Гатчина и Гатчинском районе?
Какие виды флоры и фауны, находящиеся под угрозой исчезновения, зарегистрированы в природно‑экологическом каркасе г. Гатчина и Гатчинского района?
Какие типы почв (подзолы, серные, торфяные, суглинки и т.п.) преобладают на территории г. Гатчина 

100%|██████████| 5/5 [04:08<00:00, 49.60s/it]


Транспортный анализ г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: Городской и внешний пассажирский и грузовой транспорт
Анализ инженерной инфраструктуры г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: Теплоснабжение
Какова общая протяжённость теплотрасс в г. Гатчина и в Гатчинском муниципальном районе?
Сколько тепловых пунктов (техцентров, подстанций) обслуживают территорию Гатчины?
Какие типы источников тепла (котельные, ТЭЦ, геотермальные, комбинированные) используются в системе теплоснабжения Гатчины?
Какова установленная мощность всех тепловых источников, обслуживающих Гатчину, в мегаваттах?
Какой процент тепловой энергии в Гатчине генерируется за счёт ископаемого топлива (уголь, газ, мазут) и каков процент возобновляемых источников (биоэнергетика, геотермальная энергия, солнечное тепло)?


100%|██████████| 5/5 [03:31<00:00, 42.22s/it]


In [11]:
import pickle

with open('results.pickle', 'wb') as file:
    pickle.dump(results, file)

In [12]:
with open('results.pickle', 'rb') as file:
    loaded_dict = pickle.load(file)

In [15]:
results = {k:v.content for k,v in results.items()}

In [52]:
GOAL_PROMPT = """
Ты — эксперт по стратегическому планированию территорий.
На основе полученных данных сформулируй:
- внешние и внутренние факторы развития,
- миссию,
- цели пространственного развития.

Правила работы:
- НЕ ОТВЕЧАЙ на основе собственных знаний. Используй данные.
- НЕ КОММЕНТИРУЙ пропуски в данных. Отвечай на основе того, что доступно.
- Отвечай ТОЛЬКО на русском языке.
- ИЗБЕГАЙ конкретных числовых показателей.
- Цели должны быть достижимы, но не обязаны быть соизмеримыми в текущей формулировке.
"""

GOAL_MESSAGE = """
Данные:
{data}
"""

class GoalResponse(BaseModel):
    external_factors : list[str] = Field(min_length=2, description='Внешние факторы развития')
    internal_factors : list[str] = Field(min_length=2, description='Внутренние факторы развития')
    mission : str = Field(description='Миссия')
    goals : list[str] = Field(min_length=3, max_length=5, description='Цели пространственного развития')
    # tasks : list[str] = Field(description='Задачи пространственного развития')

goal_agent = Agent(system_prompt=GOAL_PROMPT, response_format=GoalResponse, debug=True)
res = goal_agent.invoke(GOAL_MESSAGE.format(data=results))

[values] {'messages': [HumanMessage(content="\nДанные:\n{'Социально-экономический анализ г. Гатчина, (Гатчинский муниципальный район/округ, Ленинградская область). Фокус: Демография': '**Демографический профиль г.\\u202fГатчина и Гатчинского муниципального района (Ленинградская область)**  \\n\\n---\\n\\n### 1. Возрастная структура (данные «Положения о территориальном планировании Гатчинского муниципального района», 2023\\u202fг.)\\n\\n| Территория | 0‑14\\u202fлет | 15‑64\\u202fлет | 65\\u202fлет и старше |\\n|------------|----------|----------|-----------------|\\n| **Город Гатчина** | 13,0\\u202f% (≈\\u202f29,1\\u202fтыс. чел.) | 62,7\\u202f% (≈\\u202f140,2\\u202fтыс. чел.) | 24,3\\u202f% (≈\\u202f54,3\\u202fтыс. чел.) |\\n| **Гатчинский муниципальный район (включая город)** | 16,0\\u202f% (≈\\u202f41,8\\u202fтыс. чел.) | 56,0\\u202f% (≈\\u202f146,2\\u202fтыс. чел.) | 28,0\\u202f% (≈\\u202f73,0\\u202fтыс. чел.) |\\n\\n*Показатели отражают долю каждой возрастной группы от общего числ

In [53]:
res.model_dump()

{'external_factors': ['Рост населения и миграционные потоки усиливают спрос на питьевую воду, канализацию и утилизацию отходов',
  'Увеличение доли промышленного и сельскохозяйственного производства повышает нагрузку на сточные воды и требуемый объём очистки',
  'Изменения климата (повышение температуры, изменение режима осадков) влияют на доступность поверхностных и подземных вод, а также на частоту паводков',
  'Развитие транспортных коридоров и новых жилых массивов требует расширения сетей водоснабжения, канализации и электроснабжения',
  'Ужесточение экологических нормативов и требований к качеству питьевой воды и очистке сточных вод',
  'Рост интереса к возобновляемым источникам энергии (солнечная, ветровая) и к энергоэффективным системам отопления/охлаждения',
  'Увеличение объёмов бытовых и промышленных отходов требует модернизации систем их сбора и утилизации',
  'Развитие цифровых технологий (смарт‑сети, датчики, GIS) открывает возможности для более точного мониторинга и управ